In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_PATH = "../outputs/cleaned_data.csv"
MODEL_OUT = "../models/best_lead_time_model.joblib"

In [2]:
df = pd.read_csv(DATA_PATH)

CATEGORICAL_FEATURES = ["Product Name", "Factory", "Region", "Ship Mode", "Division"]
NUMERIC_FEATURES = ["Shipping_Distance_KM", "Units"]
TARGET = "Lead Time"

X = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df[TARGET]

X.head()

,Product Name,Factory,Region,Ship Mode,Division,Shipping_Distance_KM,Units
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,Interior,Standard Class,Chocolate,1563.505335,2
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Interior,Standard Class,Chocolate,1160.081298,2
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Interior,Standard Class,Chocolate,2188.313441,3
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Interior,Standard Class,Chocolate,2188.313441,3
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Atlantic,Standard Class,Chocolate,1008.215443,3


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("num", StandardScaler(), NUMERIC_FEATURES),
    ]
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(8036, 7) (2009, 7)


In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42
    ),
}

In [7]:
results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"{name:20s} | RMSE: {rmse:.3f}  MAE: {mae:.3f}  R2: {r2:.3f}")
    results.append({"model": name, "rmse": rmse, "mae": mae, "r2": r2})
    fitted_pipelines[name] = pipe

Linear Regression    | RMSE: 0.672  MAE: 0.534  R2: 0.895
Random Forest        | RMSE: 0.684  MAE: 0.544  R2: 0.892
Gradient Boosting    | RMSE: 0.677  MAE: 0.538  R2: 0.894


In [8]:
results_df = pd.DataFrame(results).sort_values("rmse")
results_df

,model,rmse,mae,r2
0,Linear Regression,0.672206,0.533924,0.895373
2,Gradient Boosting,0.677116,0.537670,0.893839
1,Random Forest,0.683827,0.544090,0.891724


In [ ]:
best_name = results_df.iloc[0]["model"]
best_pipeline = fitted_pipelines[best_name]
print(f"Best model: {best_name}")

joblib.dump(
    {
        "pipeline": best_pipeline,
        "model_name": best_name,
        "categorical_features": CATEGORICAL_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "metrics": results_df.to_dict(orient="records"),
    },
    MODEL_OUT,
)
print(f"Saved to {MODEL_OUT}")